In [1]:
# ================================================================
#  LCCN-R22a - LEAVE-ONE-CONDITION-OUT ABLATION: ARMS, ACCOUNTING,
#  CLASSICAL MACRO-F1  [CPU, ~2.5 h]
#  Answers IJIES Reviewer 2, point 2 (and feeds the new Table 6).
#  Nine arms, identical cascade and step order (pipeline v5.3):
#    loo_full          all six conditions           (== lccn)
#    loo_no_emotion    full minus emotion veto
#    loo_no_negation   full minus negation veto
#    loo_no_length     full minus length veto
#    loo_no_ortho      full minus orthographic guard
#    loo_no_gtest      full minus distributional G-test
#    loo_no_cluster    full minus cluster coherence
#    loo_vetoes_only   three vetoes, no guards      (== lccn_base)
#    loo_none          no conditions                (== ab1_none)
#  For each: unified accounting (training split first, then val and
#  test), then Logistic Regression and SVM via trainer v1.4.
#  Writes loo_texts.csv (id + text_loo_* for all 40,000 comments) for
#  the GPU notebook LCCN-R22b, loo_accounting.csv and results CSV.
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml
#             (output, trainer.py), final-emotion-dataset, gold-ds.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 4.6 MB/s eta 0:00:00
python 3.12.13


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - SWITCHABLE VETO AND THE NINE LEAVE-ONE-OUT ARMS
#  veto_flags reproduces make_veto(F2E, LEX) at theta = 3 with each of
#  the three vetoes individually switchable (same form as the
#  certified veto_theta used in lccn-sensitivity Cell 3).
# ================================================================
from collections import defaultdict, Counter
from tqdm.auto import tqdm

def veto_flags(use_emotion=True, use_negation=True, use_length=True, theta=3):
    def veto(w, c, prev=None):
        if use_emotion:
            a, b = LEX.emotion_in_context(w, prev), LEX.emotion_in_context(c, prev)
            if a and b and len(a) == 1 and len(b) == 1 and a != b: return "emotion"
        if use_negation and ((w in pipeline.NEG) != (c in pipeline.NEG)): return "negation"
        if use_length and abs(len(w) - len(c)) > theta: return "length"
        return None
    return veto

G = dict(use_gtest=True, use_cluster=True, use_ortho=True, use_veto=True)
LOO = [
    ("loo_full",        dict(),                           dict()),
    ("loo_no_emotion",  dict(),                           dict(use_emotion=False)),
    ("loo_no_negation", dict(),                           dict(use_negation=False)),
    ("loo_no_length",   dict(),                           dict(use_length=False)),
    ("loo_no_ortho",    dict(use_ortho=False),            dict()),
    ("loo_no_gtest",    dict(use_gtest=False),            dict()),
    ("loo_no_cluster",  dict(use_cluster=False),          dict()),
    ("loo_vetoes_only", dict(use_gtest=False, use_cluster=False, use_ortho=False), dict()),
    ("loo_none",        dict(use_gtest=False, use_cluster=False, use_ortho=False, use_veto=False), dict()),
]
def build(gkw, vkw):
    kw = {**G, **gkw}
    return LCCN(norm_map, evidence, veto_flags(**vkw), translit=None, lex=LEX, **kw)

single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
train_mask = df[ID].isin(SPLIT["train"]).values
df_tr, df_rest = df[train_mask], df[~train_mask]
assert len(df_tr) == 28000 and len(df_rest) == 12000

def accounting(name, o):
    tr_out = [o.normalize(t) for t in tqdm(df_tr[TXT], desc=name, leave=False)]
    keys_merged_tr = o.stats["merged"]; keys_total_tr = o.stats["merged"] + o.stats["kept"]
    blocked_tr = {k: v for k, v in o.stats.items() if k.startswith("blocked")}
    forms_rewritten_tr = len({k[1] for k, v in o.cache.items() if v != k[1]})
    forms_total_tr = len({k[1] for k in o.cache})
    rest_out = [o.normalize(t) for t in df_rest[TXT]]
    ch = tot = 0
    for a, b in zip(list(df_tr["text_raw"]) + list(df_rest["text_raw"]), tr_out + rest_out):
        ra, rb = a.lower().split(), b.lower().split()
        assert len(ra) == len(rb), f"token count mismatch in {name}"
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    mp = defaultdict(Counter)
    for a, b in zip(df_tr["text_raw"], tr_out):
        for w, x in zip(a.split(), b.split()): mp[w.lower()][x.lower()] += 1
    buckets = defaultdict(set)
    for f, c in mp.items(): buckets[c.most_common(1)[0][0]].add(f)
    pairs = lab = same = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2: continue
        pairs += n*(n-1)//2
        ce = Counter(single[f] for f in forms if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x_cross = lab - same; prec = 1 - x_cross/lab if lab else float("nan")
    full = pd.Series(tr_out + rest_out, index=list(df_tr.index) + list(df_rest.index)).reindex(df.index)
    row = dict(arm=name, tokens_changed_full_pct=round(100*ch/tot, 2),
               forms_total_train=forms_total_tr, forms_rewritten_train=forms_rewritten_tr,
               keys_merged_train=keys_merged_tr, keys_total_train=keys_total_tr,
               merged_pairs_train=pairs, labelled_pairs_train=lab, x_cross_train=x_cross,
               merge_precision_train=round(prec, 4), **blocked_tr)
    print(f"{name:<17} tok {row['tokens_changed_full_pct']:5.2f}%  forms_rew {forms_rewritten_tr:>6,}  "
          f"pairs {pairs:>6,}  X {x_cross:>4}  prec {prec:.4f}  {blocked_tr}")
    return row, full

rows = []
for name, gkw, vkw in LOO:
    row, texts = accounting(name, build(gkw, vkw))
    df[f"text_{name}"] = texts.values
    rows.append(row)
acc = pd.DataFrame(rows)
acc.to_csv("/kaggle/working/loo_accounting.csv", index=False)
print("\n", acc.to_string(index=False))

cols = [ID] + [f"text_{n}" for n, *_ in LOO]
df[cols].to_csv("/kaggle/working/loo_texts.csv", index=False, encoding="utf-8-sig")
print("\nwritten loo_texts.csv:", len(df), "rows,", len(cols)-1, "arms")
A = acc.set_index("arm")
print("\nCONSISTENCY: loo_full must equal the certified operating point (14.16%, 4,680, 2,050, X 1)")
print(f"  loo_full: {A.loc['loo_full','tokens_changed_full_pct']}%  {int(A.loc['loo_full','forms_rewritten_train']):,}  "
      f"{int(A.loc['loo_full','merged_pairs_train']):,}  X {int(A.loc['loo_full','x_cross_train'])}")
print(f"  loo_vetoes_only must equal lccn_base (19.16%, 6,282, 3,121, X 9): "
      f"{A.loc['loo_vetoes_only','tokens_changed_full_pct']}%  {int(A.loc['loo_vetoes_only','forms_rewritten_train']):,}  "
      f"{int(A.loc['loo_vetoes_only','merged_pairs_train']):,}  X {int(A.loc['loo_vetoes_only','x_cross_train'])}")


loo_full:   0%|          | 0/28000 [00:00<?, ?it/s]

loo_full          tok 14.16%  forms_rew  4,680  pairs  2,050  X    1  prec 0.9988  {'blocked_orthographic': 3321, 'blocked_gtest': 252, 'blocked_veto_negation': 35, 'blocked_veto_emotion': 102, 'blocked_cluster': 32, 'blocked_veto_length': 3}


loo_no_emotion:   0%|          | 0/28000 [00:00<?, ?it/s]

loo_no_emotion    tok 14.24%  forms_rew  4,687  pairs  2,057  X    8  prec 0.9903  {'blocked_orthographic': 3321, 'blocked_gtest': 266, 'blocked_veto_negation': 35, 'blocked_cluster': 35, 'blocked_veto_length': 3}


loo_no_negation:   0%|          | 0/28000 [00:00<?, ?it/s]

...

loo_no_cluster    tok 14.18%  forms_rew  4,688  pairs  2,055  X    1  prec 0.9988  {'blocked_orthographic': 3321, 'blocked_gtest': 252, 'blocked_veto_negation': 35, 'blocked_veto_emotion': 102, 'blocked_veto_length': 3}


loo_vetoes_only:   0%|          | 0/28000 [00:00<?, ?it/s]

loo_vetoes_only   tok 19.16%  forms_rew  6,282  pairs  3,121  X    9  prec 0.9899  {'blocked_veto_negation': 39, 'blocked_veto_emotion': 170, 'blocked_veto_length': 4}


loo_none:   0%|          | 0/28000 [00:00<?, ?it/s]

loo_none          tok 19.67%  forms_rew  6,315  pairs  3,164  X   33  prec 0.9633  {}

             arm  tokens_changed_full_pct  forms_total_train  forms_rewritten_train  keys_merged_train  keys_total_train  merged_pairs_train  labelled_pairs_train  x_cross_train  merge_precision_train  blocked_orthographic  blocked_gtest  blocked_veto_negation  blocked_veto_emotion  blocked_cluster  blocked_veto_length
       loo_full                    14.16              42456                   4680               4796             43255                2050                   821              1                 0.9988                3321.0          252.0                   35.0                 102.0             32.0                  3.0
 loo_no_emotion                    14.24              42456                   4687               4820             43255                2057                   826              8                 0.9903                3321.0          266.0                   35.0             

In [5]:
# ================================================================
#  CELL 4 - CLASSICAL MACRO-F1 (trainer v1.4, byte-identical from NB2)
#  LR and SVM on the nine arms, raw and lccn from the NB1 splits as anchors.
# ================================================================
import shutil, glob, os, sys, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "trainer.py not found - attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
sys.modules.pop("trainer", None)
import trainer; importlib.reload(trainer)
from trainer import *
assert trainer.TRAINER_VERSION == "1.4", trainer.TRAINER_VERSION
trainer.fingerprint()

sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
new_cols = [f"text_{n}" for n, *_ in LOO]
lut = {c: dict(zip(df[ID].astype(str), df[c])) for c in new_cols}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in new_cols:
        f[c] = ids.map(lut[c]); assert f[c].notna().all(), f"id join failed for {c}"
ARMS = ["raw", "lccn", "lccn_base"] + [n for n, *_ in LOO]
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM"], tag="lccn_r22_loo", resume=False)
r_lr, r_svm = res.iloc[0], res.iloc[1]
print("\n=== macro-F1 (LR, SVM, mean) ===")
print(f"{'arm':<18}{'LR':>8}{'SVM':>8}{'mean':>8}{'mean - loo_full':>18}")
ref = (float(r_lr["loo_full"]) + float(r_svm["loo_full"])) / 2
for a in ARMS:
    m = (float(r_lr[a]) + float(r_svm[a])) / 2
    print(f"{a:<18}{float(r_lr[a]):>8.4f}{float(r_svm[a]):>8.4f}{m:>8.4f}{m-ref:>+18.4f}")
print("\nDONE - loo_accounting.csv, loo_texts.csv, results_lccn_r22_loo.csv written.")
print("Save & Run All (Commit); then attach THIS output to LCCN-R22b.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / raw ===
  -> macro-F1 0.8033

=== Logistic Regression / lccn ===
  -> macro-F1 0.8017

=== Logistic Regres